In [0]:
from langchain_community.document_loaders import (DirectoryLoader,TextLoader)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.documents import Document
from dotenv import load_dotenv
import os

load_dotenv()
print(os.getenv("GEMINI_API_KEY") is not None)

# Delta table
df = spark.read.table("playerforge.gold.player_performance")
table_data = df.collect()
delta_table_document = []
for i in  table_data:
    delta_table_document.append(
        Document(
            page_content = str(i.asDict()),
            metadata={"source": "player_performance_stats"}
        )
    )
# Loader
loader = DirectoryLoader(
    "knowledge/",
    glob = "**/*.md",
    loader_cls= TextLoader
)

knowledge_documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

knowledge_document_chunks = splitter.split_documents(knowledge_documents)
delta_table_chunks = splitter.split_documents(delta_table_document)


embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

knowledge_vector_store = FAISS.from_documents(knowledge_document_chunks,embeddings)
delta_vector_store = FAISS.from_documents(delta_table_chunks,embeddings)

knowledge_retriever = knowledge_vector_store.as_retriever(
    search_kwargs={"k": 3}
)
delta_table_retriever = delta_vector_store.as_retriever(
    search_kwargs={"k": 3}
)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2,
)
question = input("What is the answer to my question : ")
# Retrieval
knowledge_results = knowledge_retriever.invoke(question)
delta_table_results = delta_table_retriever.invoke(question)

# Convert retrieved Documents into context
knowledge_context = "\n\n".join(
    doc.page_content
    for doc in knowledge_results
)
delta_context = "\n\n".join(
    doc.page_content
    for doc in delta_table_results
)

# Generation prompt
prompt = f"""
You are **PlayerForge**, a gaming analytics assistant.

Answer the user's question using **only the information provided in the context below**.

### Context

**Knowledge Context:**
{knowledge_context}

**Player Analytics Data:**
{delta_context}

### Question

{question}

### Instructions

* Use only the provided context to answer the question.
* Do not use outside knowledge or make assumptions.
* Give a concise, direct, and accurate answer.
* When relevant, combine information from both the knowledge context and player analytics data.
"""

# LLM generates the answer
response = llm.invoke(prompt)

print(response.content)

